# Qwen3.5-0.8B + FlyEmbedding-v3.1
## Preservation-first residual adapter

This experiment keeps **original Qwen exactly at initialization** and only accepts trained checkpoints that stay inside strict preservation constraints.

### What is new vs v3
- Exact identity check before training
- Real held-out **WikiText-2 validation**
- Hard checkpoint constraints instead of a soft weighted score
- Early stopping when Qwen drift becomes too large
- Safe checkpoint = **lowest validation CE among valid checkpoints**
- 12-prompt deterministic generation suite
- Interactive Qwen vs Fly-v3.1 comparison

Default safe constraints:
- top-1 agreement ≥ **99%**
- KL ≤ **0.005**
- embedding relative MSE ≤ **0.001**
- early stop if top-1 < **98.5%** or KL > **0.0075**

The goal is not just lower CE. The goal is:

> **Improve Qwen while remaining recognizably Qwen.**


In [ ]:
#@title 1. Setup repository and dependencies
import pathlib, subprocess, sys, importlib, torch

REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','-U',
    'transformers','accelerate','huggingface_hub','safetensors',
    'datasets','ipywidgets','pandas','matplotlib'
],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)

SRC_DIR=REPO_DIR/'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0,str(SRC_DIR))

for name in list(sys.modules):
    if name == 'tinycenn_lm' or name.startswith('tinycenn_lm.'):
        del sys.modules[name]
importlib.invalidate_caches()

for p in [
    REPO_DIR/'scripts'/'run_qwen35_flyembedding_v31.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flyembedding_v3.py'
]:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)

print('✓ FlyEmbedding-v3.1 preflight OK')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ CPU runtime detected. In Colab choose Runtime → Change runtime type → GPU.')


In [ ]:
#@title 2. Experiment configuration
BASE_MODEL='Qwen/Qwen3.5-0.8B' #@param {type:'string'}
RUN_MODE='quick' #@param ['quick','strong']
SEQ_LEN=128 #@param {type:'integer'}

FLY_NODES=256 #@param {type:'integer'}
GRAPH_STEPS=1 #@param {type:'integer'}
GRAPH_MIX_INIT=0.05 #@param {type:'number'}
MAX_RESIDUAL_SCALE=0.05 #@param {type:'number'}

LR_CORE=0.00015 #@param {type:'number'}
LR_GATE=0.00030 #@param {type:'number'}
PROBE_EVERY=25 #@param {type:'integer'}

MIN_TOP1=0.99 #@param {type:'number'}
MAX_KL=0.005 #@param {type:'number'}
MAX_EMBEDDING_MSE=0.001 #@param {type:'number'}
EARLY_STOP_TOP1=0.985 #@param {type:'number'}
EARLY_STOP_KL=0.0075 #@param {type:'number'}

OUTPUT_DIR=REPO_DIR/'results'/'flyembedding_v31_qwen35_08b'

print('Mode:',RUN_MODE)
print('Safe checkpoint constraints:')
print(' top1 >=',MIN_TOP1)
print(' KL   <=',MAX_KL)
print(' emb  <=',MAX_EMBEDDING_MSE)
print('Early stop:')
print(' top1 <',EARLY_STOP_TOP1,'or KL >',EARLY_STOP_KL)


In [ ]:
#@title 3. Run FlyEmbedding-v3.1 — live output
import subprocess, sys

cmd=[
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_flyembedding_v31.py'),
    '--base-model',BASE_MODEL,
    '--run-mode',RUN_MODE,
    '--seq-len',str(SEQ_LEN),
    '--fly-nodes',str(FLY_NODES),
    '--graph-steps',str(GRAPH_STEPS),
    '--graph-mix-init',str(GRAPH_MIX_INIT),
    '--max-residual-scale',str(MAX_RESIDUAL_SCALE),
    '--lr-core',str(LR_CORE),
    '--lr-gate',str(LR_GATE),
    '--probe-every',str(PROBE_EVERY),
    '--min-top1',str(MIN_TOP1),
    '--max-kl',str(MAX_KL),
    '--max-embedding-mse',str(MAX_EMBEDDING_MSE),
    '--early-stop-top1',str(EARLY_STOP_TOP1),
    '--early-stop-kl',str(EARLY_STOP_KL),
    '--output-dir',str(OUTPUT_DIR),
]

print('='*110)
print(' '.join(cmd))
print('='*110)

p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in iter(p.stdout.readline,''):
    print(line,end='',flush=True)
rc=p.wait()
print('\nFinished, exit code',rc)
if rc:
    raise subprocess.CalledProcessError(rc,cmd)


In [ ]:
#@title 4. Summary and safe-checkpoint result
import json, pandas as pd
from IPython.display import display

report=json.loads((OUTPUT_DIR/'report.json').read_text())
hist=pd.read_csv(OUTPUT_DIR/'training_history.csv')
probes=pd.read_csv(OUTPUT_DIR/'validation_probes.csv')

print('Architecture:',report['architecture'])
print('Data source:',report['data_source'])
print('Selected safe checkpoint step:',report['selected_step'])
print('Stop reason:',report['stop_reason'])
print('Identity:',report['identity_embedding_check'])
print('Parameter stats:')
print(json.dumps(report['parameter_stats'],indent=2))
print('\nInitial validation:')
print(json.dumps(report['initial_probe'],indent=2))
print('\nSelected/final validation:')
print(json.dumps(report['final_probe'],indent=2))
print('\nGeneration summary:')
print(json.dumps(report['generation_summary'],indent=2))
print('\nQUALITY GATE:',report['quality_gate_passed'])

print('\nValidation probes:')
display(probes)


In [ ]:
#@title 5. Plot preservation vs validation improvement
import matplotlib.pyplot as plt

fig=plt.figure(figsize=(9,5))
plt.plot(probes['step'],probes['student_ce'],marker='o',label='Fly-v3.1 CE')
plt.plot(probes['step'],probes['teacher_ce'],marker='o',label='Qwen CE')
plt.axvline(report['selected_step'],linestyle='--',label='selected safe checkpoint')
plt.xlabel('Training step')
plt.ylabel('Validation CE')
plt.title('Held-out validation CE')
plt.legend()
plt.show()

fig=plt.figure(figsize=(9,5))
plt.plot(probes['step'],probes['top1_logit_agreement'],marker='o',label='Top-1 agreement')
plt.axhline(MIN_TOP1,linestyle='--',label='safe minimum')
plt.axhline(EARLY_STOP_TOP1,linestyle=':',label='early-stop boundary')
plt.xlabel('Training step')
plt.ylabel('Agreement')
plt.title('Qwen preservation')
plt.legend()
plt.show()

fig=plt.figure(figsize=(9,5))
plt.plot(probes['step'],probes['teacher_kl'],marker='o',label='KL vs Qwen')
plt.axhline(MAX_KL,linestyle='--',label='safe maximum')
plt.axhline(EARLY_STOP_KL,linestyle=':',label='early-stop boundary')
plt.xlabel('Training step')
plt.ylabel('KL')
plt.title('Distribution drift')
plt.legend()
plt.show()


In [ ]:
#@title 6. 12-prompt deterministic generation comparison
rows=report['generation_samples']
for i,x in enumerate(rows,1):
    print('\n'+'='*100)
    print(f'{i}. USER:',x['prompt'])
    print('\nQWEN:',x['qwen_reply'])
    print('\nFLY :',x['fly_reply'])
    print(
        '\nexact=',x['exact_token_match'],
        '| prefix=',x['matching_prefix_tokens'],
        '| jaccard=',round(x['token_jaccard'],3),
        '| passed=',x['passed']
    )


In [ ]:
#@title 7. Reload selected v3.1 adapter and compare interactively
import torch, ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_flyembedding_v3 import FlyEmbeddingV3Config, install_fly_embedding_v3

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)

tok=AutoTokenizer.from_pretrained(BASE_MODEL,use_fast=True)
if tok.pad_token_id is None:
    tok.pad_token=tok.eos_token

qwen=AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype,low_cpu_mem_usage=True).to(device).eval()
fly=AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype,low_cpu_mem_usage=True).to(device).eval()

def make_adj(n):
    a=torch.zeros(n,n,dtype=torch.float32)
    for i in range(n):
        a[i,i]=1
        for s in (1,3,7,17):
            a[i,(i+s)%n]=1
            a[i,(i-s)%n]=1
    return (a/a.sum(-1,keepdim=True).clamp_min(1)).to(device)

ckpt=torch.load(OUTPUT_DIR/'fly_embedding_v31_adapter.pt',map_location='cpu')
cfg=FlyEmbeddingV3Config(**ckpt['config'])
install_fly_embedding_v3(fly,cfg,make_adj(cfg.fly_nodes))
fly.fly_embedding_v3_core.load_state_dict(ckpt['fly_embedding_v3_core'],strict=True)
fly.eval()

def answer(model,prompt):
    text=tok.apply_chat_template(
        [{'role':'user','content':prompt}],
        tokenize=False,add_generation_prompt=True
    )
    enc=tok(text,return_tensors='pt').to(device)
    with torch.no_grad():
        y=model.generate(
            **enc,max_new_tokens=128,do_sample=False,use_cache=True,
            pad_token_id=tok.eos_token_id
        )
    return tok.decode(y[0,enc.input_ids.shape[1]:],skip_special_tokens=True).strip()

prompt_box=widgets.Textarea(
    value='Explain in simple terms why residual adapters are useful.',
    description='Prompt:',
    layout=widgets.Layout(width='100%',height='100px')
)
button=widgets.Button(description='Compare Qwen vs Fly-v3.1',button_style='primary')
out=widgets.Output()

def run_compare(_):
    q=prompt_box.value.strip()
    if not q:
        return
    with out:
        clear_output()
        print('QWEN\n-----\n'+answer(qwen,q))
        print('\n'+'='*100+'\n')
        print('FLY-v3.1\n--------\n'+answer(fly,q))

button.on_click(run_compare)
display(prompt_box,button,out)


## How to interpret the result

A strong result is **not** simply a very negative CE gap.

The useful region is where:
- validation CE is lower than Qwen,
- top-1 agreement stays ≥ 99%,
- KL stays ≤ 0.005,
- deterministic outputs remain coherent and mostly close to Qwen,
- the selected checkpoint occurs **before** drift accelerates.

If the best safe checkpoint is around step 75–125, that is expected based on the previous v3 run.
